"En este proyecto estoy trabajando con un conjunto de datos sobre Meningitis. Mi objetivo principal es entrenar a un modelo matemático para que aprenda a predecir el nivel de glucosa (Glucose_Level) de un paciente, basándose exclusivamente en sus análisis de sangre y características biológicas (como su edad, glóbulos blancos, proteínas, hemoglobina, etc.).
la variable objetivo es predecir el nivel de glucosa de un paciente usando sus análisis de sangre.
Para limpiar mis datos, primero elimino las filas que no tienen el dato de la glucosa. Después, borro el Patient_ID porque es un simple número de registro sin valor médico, y elimino columnas como Diagnosis o Outcome. Es crucial quitar estas últimas para evitar que la Inteligencia Artificial haga trampa copiando el diagnóstico final del médico en lugar de aprender a predecirlo.
Finalmente, convierto los textos a números, relleno los datos faltantes con la mediana y separo la glucosa del resto de los análisis.

In [21]:
# FASE 1: LIBRERÍAS Y CARGA DE DATOS LOCALES
import os
import numpy as np
import pandas as pd
from matplotlib import pyplot

%matplotlib inline

# 1. Tu ruta local exacta en Mac (sin las barras invertidas)
ruta_meningitis = '/Users/mac/Desktop/datasets/Meningitis Dataset with Missing Values/mening missing 12.csv'

# 2. Cargamos el archivo CSV a la memoria (creamos df_crudo)
df_crudo = pd.read_csv('/Users/mac/Desktop/datasets/ Meningitis Dataset with Missing Values/mening missing 12.csv')

print(" Fase 1: Entorno listo y datos cargados desde tu Mac.")
print(f"Total de registros crudos cargados: {len(df_crudo)}")

 Fase 1: Entorno listo y datos cargados desde tu Mac.
Total de registros crudos cargados: 1200


In [22]:
import os
import numpy as np
import pandas as pd
from matplotlib import pyplot
import kagglehub

%matplotlib inline
print(" Fase 1: Entorno listo.")

 Fase 1: Entorno listo.


In [31]:
# FASE 2.1: ELIMINACIÓN DE COLUMNAS
df = df_crudo.copy()
df.columns = df.columns.str.replace(' ', '')

# Dropeamos según la auditoría: Glucosa es la Y
cols_a_eliminar = ['Patient_ID', 'Diagnosis', 'Risk_Level', 'Outcome']
df = df.drop(columns=cols_a_eliminar, errors='ignore')

# LIMPIEZA EXTRA: Eliminar filas donde la Y (Glucosa) sea nula de origen
df = df.dropna(subset=['Glucose_Level'])

print(f" Columnas eliminadas. Filas restantes: {len(df)}")
print("DATASET PEINADO:")
display(df.head(5))

 Columnas eliminadas. Filas restantes: 1192
DATASET PEINADO:


,Age,Gender,WBC_Count,Protein_Level,Glucose_Level,Pathogen_Present,Hemoglobin,WBC_Blood_Count,Platelets,CRP_Level
0,101.0,Female,8624.0,16.0,83.0,No,15.0,7269.0,160949.0,71.0
1,78.0,Male,22623.0,200.0,41.0,No,18.0,6532.0,371741.0,41.0
2,8.0,Female,12908.0,39.0,3.0,No,16.0,7417.0,180403.0,22.0
3,104.0,Female,15072.0,58.0,36.0,Yes,7.0,13792.0,132254.0,48.0
4,38.0,Female,18623.0,152.0,34.0,Yes,5.0,17054.0,134941.0,28.0


In [25]:
# 1. Convertir categorías (texto) a números
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = pd.factorize(df[col])[0]

# 2. Rellenar nulos: Agregamos numeric_only=True para evitar el TypeError
# Esto hace que la mediana ignore las columnas que aún tengan texto
df = df.fillna(df.median(numeric_only=True))

# 3. Eliminar columnas que se volvieron constantes
df = df.loc[:, (df != df.iloc[0]).any()]

In [33]:
# --- CORRECCIÓN ANTES DE NORMALIZAR ---

# 1. Nos aseguramos de que TODO en el DataFrame sea numérico
# Si alguna columna no se pudo convertir a número, se transformará en NaN
df = df.apply(pd.to_numeric, errors='coerce')

# 2. Volvemos a limpiar cualquier nulo que haya aparecido por la conversión
df = df.fillna(df.median(numeric_only=True))

# 3. Ahora sí, separamos X e Y
X_raw = df.drop(columns=['Glucose_Level']).values
y_raw = df['Glucose_Level'].values.reshape(-1, 1)

# 4. Normalizar (Ya no debería dar error)
mu_x, sigma_x = np.mean(X_raw, axis=0), np.std(X_raw, axis=0) + 1e-8
X_norm = (X_raw - mu_x) / sigma_x

# 5. Añadir columna de Bias
X = np.concatenate([np.ones((y_raw.size, 1)), X_norm], axis=1)

print("Normalización completada con éxito.")


Normalización completada con éxito.
